# Ubicación de cámaras de vigilancia mediante Vertex Cover

Notebook de experimentos del proyecto. Orden de trabajo:

1. Modelado (grilla pura y perturbada)
2. Complejidad: certificado y reducción a Independent Set
3. Algoritmos sobre una instancia
4. Instancias chicas (razón de aproximación real)
5. Instancias medianas (cota inferior y escalabilidad)
6. Umbral de perturbación (desafío extra)
7. Vertex Cover ponderado (desafío extra)

> Los parámetros de este notebook son **orientativos**. Elegí los tuyos y justificalos en el informe.

In [ ]:
import sys
sys.path.insert(0, ".")          # para importar los módulos de esta carpeta

import matplotlib.pyplot as plt
import pandas as pd

from grafos import *
from algoritmos import *
from complejidad import *
from experimentos import *
from visualizacion import *

pd.set_option("display.width", 140)

## 1. Modelado

Nodos = intersecciones, aristas = calles. Una cámara en una intersección vigila todas las calles que llegan a ella (supuesto de cobertura de 360°), así que el problema es Vertex Cover.

La **grilla pura** es bipartita (König: óptimo en tiempo polinomial). En la **grilla perturbada**, basta una diagonal para generar un ciclo impar y dejar de ser bipartita.

In [ ]:
pura = grilla_pura(6, 6)
pert = grilla_perturbada(6, 6, p_d=0.25, p_q=0.05, seed=42)

print("Pura      -> aristas:", cantidad_aristas(pura), "| bipartita:", es_bipartito(pura))
print("Perturbada-> aristas:", cantidad_aristas(pert), "| diagonales:", contar_diagonales(pert),
      "| bipartita:", es_bipartito(pert), "| conexa:", es_conexo(pert))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
dibujar_grilla(pura, titulo="Grilla pura (control)", ax=axes[0])
dibujar_grilla(pert, titulo="Grilla perturbada (azul = diagonales)", ax=axes[1])
plt.show()

**Para el informe:** verificá qué pasa con `p_d = 0` y `p_q > 0` (¿sigue siendo bipartita?) y con una sola diagonal.

In [ ]:
g = grilla_perturbada(6, 6, p_d=0.0, p_q=0.2, seed=1)
print("p_d=0, p_q=0.2 -> bipartita:", es_bipartito(g), "| diagonales:", contar_diagonales(g))

## 2. Complejidad: certificado y reducción Vertex Cover ↔ Independent Set

* Certificado: un conjunto `S`. Verificador polinomial: `es_cobertura(G, S, K)`.
* Lema: `S` es cobertura **si y solo si** `N \ S` es independiente. La transformación es `(G, K) → (G, |N| − K)`.

In [ ]:
chica = grilla_perturbada(4, 4, p_d=0.3, p_q=0.05, seed=3)

cobs = {"greedy": greedy_grado_max(chica),
        "2-aprox": matching_maximal_2aprox(chica)[0]}
for nombre, S in cobs.items():
    print(f"{nombre:8s} es cobertura: {es_cobertura(chica, S)} | "
          f"su complemento es independiente: {es_independiente(chica, complemento(chica, S))}")

print()
print(verificar_reduccion(chica, cobs))

In [ ]:
# Verificación sobre muchas instancias chicas: tau(G) + alpha(G) = |N|
ok = 0
total = 0
for seed in range(20):
    g = grilla_perturbada(4, 4, p_d=0.3, p_q=0.1, seed=seed)
    r = verificar_reduccion(g)
    total += 1
    ok += r["tau_mas_alpha_igual_n"] and r["fuerza_bruta_coincide"]
print(f"{ok}/{total} instancias cumplen tau + alpha = n")

## 3. Los tres algoritmos sobre una instancia

Greedy por grado máximo, 2-aproximación por matching maximal y óptimo exacto (König si es bipartita, branching en general).

In [ ]:
inst = grilla_perturbada(6, 6, p_d=0.25, p_q=0.05, seed=42)
opt = branching_exacto(inst)

soluciones = {
    "Óptimo (branching)": opt["cobertura"],
    "Greedy": greedy_grado_max(inst),
    "2-aprox (orden fijo)": matching_maximal_2aprox(inst)[0],
    "2-aprox (aleatorio)": matching_maximal_2aprox(inst, "aleatorio", seed=1)[0],
}
for nombre, cob in soluciones.items():
    assert es_cobertura(inst, cob)
    print(f"{nombre:22s} {len(cob):3d} cámaras  (razón {len(cob)/opt['tam']:.2f})")
print("Cota inferior (matching máximo):", cota_inferior(inst))
print("Nodos explorados por el branching:", opt["nodos"], "| tiempo:", round(opt["tiempo"], 4), "s")

comparar_coberturas(inst, soluciones)
plt.show()

## 4. Instancias chicas: razón de aproximación real

Acá el óptimo exacto se calcula siempre, así que `razon_real = tam / óptimo`.

In [ ]:
chicas = barrido(
    tamanos=[(4, 4), (5, 5), (6, 6)],
    p_ds=[0.0, 0.05, 0.1, 0.2, 0.4],
    p_q=0.05,
    replicas=10,            # al menos 10 por configuración
    semilla_base=2026,      # registrá la semilla en el informe
    timeout=10,
    repeticiones=3,
    verbose=False,
)
assert chicas["valida"].all()
chicas.to_csv("resultados_chicas.csv", index=False)

resumen = resumir(chicas, ["p_d", "algoritmo"], ("tam", "razon_real", "tiempo_s"))
resumen

In [ ]:
graficar_razon_vs_pd(chicas, "razon_real")
plt.show()

## 5. Instancias medianas: cota inferior y escalabilidad

El branching puede no terminar: se fija un `timeout` y se registra en `completo`. Si no terminó, la razón se mide contra la cota inferior (`razon_cota`), que es una **cota superior** de la razón real. En la grilla pura, König da el óptimo exacto en cualquier tamaño.

In [ ]:
medianas = barrido(
    tamanos=[(8, 8), (10, 10), (12, 12), (15, 15)],
    p_ds=[0.0, 0.05, 0.1, 0.2, 0.4],
    p_q=0.05,
    replicas=5,             # subí a 10 para el informe final
    semilla_base=2026,
    timeout=5,              # subí si tu compu lo permite
    repeticiones=1,
    verbose=True,
)
assert medianas["valida"].all()
medianas.to_csv("resultados_medianas.csv", index=False)

resumir(medianas, ["n", "p_d", "algoritmo"], ("tam", "razon_cota", "tiempo_s"))

In [ ]:
graficar_razon_vs_pd(medianas, "razon_cota", tam=(15, 15))
graficar_tiempos(medianas)
plt.show()

## 6. Desafío extra: umbral de perturbación

¿Cuántas diagonales hacen falta para que el branching se dispare y las heurísticas se alejen del óptimo?

In [ ]:
graficar_branching_vs_pd(medianas, tam=(15, 15))
plt.show()

umbral = (medianas[medianas.algoritmo == "branching"]
          .groupby(["n", "p_d"])
          .agg(tiempo_medio=("tiempo_s", "mean"),
               resueltas=("completo", "mean"),
               nodos_arbol=("nodos_arbol", "mean"))
          .round(3))
umbral

## 7. Desafío extra: Vertex Cover ponderado

Cada intersección tiene un costo de instalación. Se compara el greedy ponderado, la 2-aproximación por *local ratio* (con y sin poda de redundantes) y el branching ponderado, y se mide cuánto cuesta usar una solución que **ignora** los costos.

In [ ]:
pond = barrido_ponderado(
    tamanos=[(5, 5), (6, 6)],
    p_ds=[0.0, 0.1, 0.4],
    p_q=0.05,
    replicas=10,
    timeout=10,
    avenidas=True,          # las avenidas (fila y columna central) cuestan x3
    verbose=False,
)
assert pond["valida"].all()
resumir(pond, ["algoritmo"], ("costo", "razon_real"))

In [ ]:
w_demo_g = grilla_perturbada(6, 6, 0.2, 0.05, seed=11)
w_demo = asignar_costos(w_demo_g, 6, 6, seed=11, avenidas=True)
r_pond = branching_exacto(w_demo_g, w=w_demo)
r_sin = branching_exacto(w_demo_g)

comparar_coberturas(w_demo_g, {
    "Ignorando costos": r_sin["cobertura"],
    "Con costos (óptimo)": r_pond["cobertura"],
}, costos=w_demo)
plt.show()
print("Costo ignorando costos:", costo_cobertura(r_sin["cobertura"], w_demo))
print("Costo óptimo ponderado:", r_pond["costo"])

## 8. Conclusiones (completar)

* ¿Cuánto se aleja cada heurística del óptimo en la grilla pura y en la perturbada?
* ¿El 2-aprox se acerca al factor 2 en la práctica? ¿Gana el greedy aunque no tenga garantía?
* ¿Cómo crece el tiempo del branching con el tamaño y con `p_d`? ¿Desde qué `p_d` deja de ser viable?
* ¿Qué pasa con `p_d = 0` y `p_q > 0`?
* ¿Cómo cambian las ubicaciones al considerar costos?